In [143]:
import numpy as np
import pandas as pd

In [144]:
df=pd.read_csv('../Dataset/Cleaned_Dataset')
df.sample()

,MainBranch,Age,EdLevel,Employment,WorkExp,LearnCodeChoose,LearnCodeAI,YearsCode,DevType,OrgSize,...,RemoteWork,PurchaseInfluence,AIThreat,NewRole,ToolCountWork,ToolCountPersonal,Country,AIModelsChoice,ConvertedCompYearly,JobSat
9296,I am a developer by profession,35-44 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,11.0,"Yes, I am not new to coding but am learning ne...","Yes, I learned how to use AI-enabled tools req...",15.0,"Developer, front-end",20 to 99 employees,...,In-person,"Yes, I endorsed a tool that was open-source an...",I'm not sure,I have somewhat considered changing my career ...,NaN,NaN,Hong Kong (S.A.R.),Yes,96816.0,7.0


In [145]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [146]:
from sklearn.preprocessing import OneHotEncoder

In [147]:
X=df.drop(columns=['ConvertedCompYearly', 'JobSat'])
y=df['ConvertedCompYearly']

In [148]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [149]:
cat_cols = X.select_dtypes(include=['object', 'category']).columns

In [150]:
num_cols=X.select_dtypes(include=['float64']).columns

In [151]:
df.skew(numeric_only=True)

WorkExp                 1.087463
YearsCode               0.919303
ToolCountWork          42.907499
ToolCountPersonal      24.297659
ConvertedCompYearly    72.968280
JobSat                 -0.991724
dtype: float64

In [152]:
num_pipeline=Pipeline([
    ('imputer',SimpleImputer(strategy='median'))
])

In [153]:
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [154]:
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

In [155]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features=1.0,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

In [156]:
model_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", rf)
])

In [157]:
model_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ('selector', VarianceThreshold(threshold=0.01)),
    ("model", rf)
])

In [158]:
model_pipeline.fit(X_train,y_train)

,steps,"[('preprocessing', ...), ('selector', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [159]:
model_pipeline.named_steps['selector'].variances_

array([9.29883733e+01, 1.14023754e+02, 2.68395712e+04, 1.05928720e+03,
       0.00000000e+00, 9.04162082e-02, 2.34187653e-01, 2.17370996e-01,
       1.20988879e-01, 5.11879833e-02, 9.98880212e-03, 8.34831166e-04,
       3.26039702e-02, 2.47955605e-01, 2.07395793e-01, 7.01988014e-03,
       4.22392399e-03, 4.08550010e-02, 4.50079657e-02, 1.03044879e-01,
       1.35340351e-01, 1.34787944e-03, 1.10055052e-01, 1.71762285e-02,
       1.34787944e-03, 1.50626265e-02, 2.05110704e-01, 8.72766171e-03,
       2.08764301e-01, 8.37232827e-02, 9.39397097e-02, 9.04162082e-02,
       2.09491938e-01, 2.40699450e-01, 1.52495050e-02, 8.47503707e-03,
       3.58624877e-03, 6.46503536e-02, 9.98880212e-03, 3.71384991e-03,
       2.06366358e-02, 2.94774738e-03, 1.06181331e-02, 2.94774738e-03,
       2.36432392e-02, 4.60613257e-03, 5.49679595e-03, 1.55479025e-01,
       5.15889871e-02, 3.51139164e-02, 5.07865747e-02, 2.31073640e-01,
       9.42169789e-03, 3.45772976e-02, 2.46817117e-02, 7.70662954e-04,
      

In [160]:
feature_names = model_pipeline.named_steps['preprocessing'].get_feature_names_out()

selected_features = feature_names[
    model_pipeline.named_steps['selector'].get_support()
]
print(selected_features)

['num__WorkExp' 'num__YearsCode' 'num__ToolCountWork'
 'num__ToolCountPersonal' 'cat__Age_18-24 years old'
 'cat__Age_25-34 years old' 'cat__Age_35-44 years old'
 'cat__Age_45-54 years old' 'cat__Age_55-64 years old'
 'cat__Age_65 years or older' 'cat__Age_Prefer not to say'
 'cat__EdLevel_Associate degree (A.A., A.S., etc.)'
 'cat__EdLevel_Bachelor’s degree (B.A., B.S., B.Eng., etc.)'
 'cat__EdLevel_Master’s degree (M.A., M.S., M.Eng., MBA, etc.)'
 'cat__EdLevel_Other (please specify):'
 'cat__EdLevel_Primary/elementary school'
 'cat__EdLevel_Professional degree (JD, MD, Ph.D, Ed.D, etc.)'
 'cat__EdLevel_Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)'
 'cat__EdLevel_Some college/university study without earning a degree'
 'cat__Employment_Employed' 'cat__Employment_I prefer not to say'
 'cat__Employment_Independent contractor, freelancer, or self-employed'
 'cat__Employment_Not employed' 'cat__Employment_Retired'
 'cat__Employment_Student'
 'cat__Le

In [161]:
removed_features = feature_names[
    ~model_pipeline.named_steps['selector'].get_support()
]
removed_features

array(['cat__MainBranch_I am a developer by profession'], dtype=object)

In [162]:
imp_feature = model_pipeline.named_steps[
    'model'
].feature_importances_
imp_feature

array([4.27776481e-02, 1.50966878e-01, 3.94308574e-02, 6.70862105e-02,
       5.73740618e-03, 1.24141348e-03, 3.80243380e-02, 2.23669163e-03,
       1.45625273e-04, 4.92895595e-05, 2.94106257e-06, 1.30647558e-04,
       6.83973524e-04, 1.17040888e-03, 3.43902978e-05, 1.87068911e-02,
       4.79999805e-03, 2.60623752e-04, 1.22601138e-03, 2.30508455e-03,
       7.61302694e-05, 5.12549255e-03, 4.16756008e-03, 5.73260998e-06,
       1.35871199e-04, 1.13049151e-02, 2.28637436e-05, 1.65393700e-02,
       1.48669173e-03, 1.73435209e-02, 3.39261192e-04, 1.43737728e-03,
       1.08221113e-02, 5.97322605e-04, 2.63160249e-05, 1.90987651e-05,
       1.42061632e-03, 2.66470921e-04, 1.09286601e-04, 3.42504958e-04,
       2.06329900e-06, 2.24901451e-03, 6.11918067e-06, 1.48529861e-04,
       1.59148100e-04, 1.19400110e-05, 2.49483870e-03, 3.13789398e-04,
       1.34991723e-04, 1.15552959e-04, 3.45024979e-02, 3.36845874e-05,
       3.07678436e-04, 8.92694766e-02, 1.71730553e-05, 2.00002067e-03,
      

## Decision tree important feature

In [163]:
imp_feature_df = pd.DataFrame({
    'Feature': selected_features,
    'Importance': imp_feature
}).sort_values(by='Importance', ascending=False)

print(imp_feature_df.head(10))

                                               Feature  Importance
1                                       num__YearsCode    0.150967
85   cat__PurchaseInfluence_Yes, I influenced the p...    0.120527
53                    cat__DevType_Engineering manager    0.089269
3                               num__ToolCountPersonal    0.067086
67                   cat__OrgSize_100 to 499 employees    0.046805
237              cat__Country_United States of America    0.044519
0                                         num__WorkExp    0.042778
2                                   num__ToolCountWork    0.039431
6                             cat__Age_35-44 years old    0.038024
50                  cat__DevType_Developer, full-stack    0.034502


In [164]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    model_pipeline,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42
)

In [165]:
print(result)

{'importances_mean': array([ 2.22044605e-16,  3.07264672e-02,  4.77454001e-03,  3.35444794e-03,
       -1.34930038e-02, -1.05587161e-03,  6.05726023e-03, -1.37401592e-01,
        3.61971635e-02,  9.89531734e-04,  2.49805466e-02,  7.65458389e-03,
       -1.24033317e-01, -4.16773233e-03, -1.06396163e-02,  1.86418344e-02,
        1.41827174e-02,  7.47202695e-01,  5.82007471e-03]), 'importances_std': array([0.        , 0.05749881, 0.01112226, 0.00180614, 0.08249091,
       0.01223588, 0.01195385, 0.08292728, 0.04384499, 0.02003279,
       0.03608072, 0.01225445, 0.13524876, 0.00391284, 0.00901882,
       0.01890293, 0.04555735, 0.3992814 , 0.00933411]), 'importances': array([[ 2.22044605e-16,  2.22044605e-16,  2.22044605e-16,
         2.22044605e-16,  2.22044605e-16,  2.22044605e-16,
         2.22044605e-16,  2.22044605e-16,  2.22044605e-16,
         2.22044605e-16],
       [ 6.50655102e-02,  6.35889899e-02,  5.09104394e-02,
        -2.99599139e-02, -3.37702649e-02, -3.66012362e-02,
      

In [166]:
imp = result['importances_mean']
imp

array([ 2.22044605e-16,  3.07264672e-02,  4.77454001e-03,  3.35444794e-03,
       -1.34930038e-02, -1.05587161e-03,  6.05726023e-03, -1.37401592e-01,
        3.61971635e-02,  9.89531734e-04,  2.49805466e-02,  7.65458389e-03,
       -1.24033317e-01, -4.16773233e-03, -1.06396163e-02,  1.86418344e-02,
        1.41827174e-02,  7.47202695e-01,  5.82007471e-03])

In [167]:
mask = model_pipeline.named_steps['selector'].get_support()
selected_features2 = feature_names[mask]

In [168]:
perm_importance_df = pd.DataFrame({
    'Feature': selected_features2[:len(imp)],
    'Feature_Importance': imp
}).sort_values(by='Feature_Importance', ascending=False)

print(perm_importance_df)

                                              Feature  Feature_Importance
17  cat__EdLevel_Secondary school (e.g. American h...        7.472027e-01
8                            cat__Age_55-64 years old        3.619716e-02
1                                      num__YearsCode        3.072647e-02
10                         cat__Age_Prefer not to say        2.498055e-02
15             cat__EdLevel_Primary/elementary school        1.864183e-02
16  cat__EdLevel_Professional degree (JD, MD, Ph.D...        1.418272e-02
11   cat__EdLevel_Associate degree (A.A., A.S., etc.)        7.654584e-03
6                            cat__Age_35-44 years old        6.057260e-03
18  cat__EdLevel_Some college/university study wit...        5.820075e-03
2                                  num__ToolCountWork        4.774540e-03
3                              num__ToolCountPersonal        3.354448e-03
9                          cat__Age_65 years or older        9.895317e-04
0                                     

In [169]:
df2 = perm_importance_df.copy()
df2['OriginalFeature'] = df2['Feature'].str.split('__').str[1].str.split('_').str[0]
grouped = df2.groupby('OriginalFeature')['Feature_Importance'].sum()

print(grouped.sort_values(ascending=False))

OriginalFeature
EdLevel              6.546612e-01
YearsCode            3.072647e-02
ToolCountWork        4.774540e-03
ToolCountPersonal    3.354448e-03
WorkExp              2.220446e-16
Age                 -8.372597e-02
Name: Feature_Importance, dtype: float64
